
# LLM Evaluation Pipeline (Math Reasoning)

This notebook provides a **ready-to-run pipeline** to run an LLM on a math reasoning dataset (AIMathsOlympiadData) and store the answer in a separate column in the df.

## What this notebook does:
- Loads dataset
- Formats prompts
- Runs LLM (placeholder / OpenAI-ready)
---

## ⚠️ Setup Instructions
1. Download dataset from Kaggle:
   https://www.kaggle.com/datasets/satyaprakashshukl/aimathsolympiaddata

2. Place the CSV/parquet, etc in:
   `data/file.csv`


In [9]:
import pandas as pd
import numpy as np
import random
import os
from typing import Dict
from dotenv import load_dotenv

# .envrc format is 'export KEY="VALUE"', so we use a custom function to load it 
# since standard load_dotenv might not handle 'export' and the specific format correctly.
def load_envrc(filepath):
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith('export '):
                    # Remove 'export ' and handle both 'KEY="VALUE"' and 'KEY: "VALUE"' formats
                    part = line[7:]
                    if '=' in part:
                        key, value = part.split('=', 1)
                    elif ': ' in part:
                        key, value = part.split(': ', 1)
                    else:
                        continue
                    
                    # Strip quotes from key and value
                    key = key.strip().strip('"').strip("'")
                    value = value.strip().strip('"').strip("'")
                    os.environ[key] = value

load_envrc('.envrc')
print("OPENAI_API_KEY loaded:", "OPENAI_API_KEY" in os.environ)


OPENAI_API_KEY loaded: True


## Recommended Metrics for Math Reasoning

For evaluating LLMs on math problems, standard text metrics (like ROUGE) are less useful than logic-based ones. Here are the recommended metrics:

1.  **Exact Match (EM)**: Did the parsed `LLM_output` match the `answer` exactly? (Highest priority).
2.  **Relative Error / Accuracy**: How close was the numerical answer?
3.  **Faithfulness/Grounding**: Does the reasoning (Chain of Thought) actually lead to the final answer, or is it a "hallucinated" correct answer?
4.  **ROUGE-L / BLEU**: To compare the similarity of the *explanation* text, though this is secondary to numerical correctness.


In [ ]:
import pandas as pd
import numpy as np
import re
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Load the specific CSV with LLM outputs
eval_df = pd.read_csv('data/dataset_math_with_llm_output.csv')

def compute_math_metrics(row):
    gt = str(row['answer']).strip()
    llm_full_text = row['LLM_output']
    
    # Use our previous parser for numerical extraction
    pred = parse_answer(llm_full_text)
    
    # 1. Exact Match (Numerical)
    try:
        gt_num = float(gt)
        em = 1.0 if np.isclose(pred, gt_num) else 0.0
    except:
        em = 1.0 if str(pred) == gt else 0.0
        
    # 2. Textual Similarity (ROUGE-L) for Reasoning Quality
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_l = scorer.score(str(gt), llm_full_text)['rougeL'].fmeasure
    
    return {
        'exact_match': em,
        'rouge_l': rouge_l,
        'parsed_answer': pred
    }

# Apply metrics
metrics_results = eval_df.apply(compute_math_metrics, axis=1, result_type='expand')
eval_results_df = pd.concat([eval_df, metrics_results], axis=1)

print("Evaluation Summary:")
print(eval_results_df[['exact_match', 'rouge_l']].mean())
eval_results_df.head()


In [2]:
df = pd.read_parquet('data/train-00000-of-00001.parquet')
print(df.shape)
df.head()

(200035, 2)


,question,answer
0,Jungkook is the 5th place. Find the number of ...,"If Jungkook is in 5th place, then 4 people cro..."
1,A number divided by 10 is 6. Yoongi got the re...,"Let's call the certain number ""x"". According t..."
2,Dongju selects a piece of paper with a number ...,To find the second smallest and third smallest...
3,"You wanted to subtract 46 from a number, but y...",If you accidentally subtracted 59 instead of 4...
4,The length of one span of Jinseo is about 12 c...,If one span of Jinseo is about 12 centimeters ...


In [5]:

def build_prompt(question: str, cot: bool = False) -> str:
    if cot:
        return f"Q: {question}\nSolve the problem, think step by step and give your answer."
    else:
        return f"Q: {question}\nA:"


In [10]:
# OpenAI API call
from openai import OpenAI
import os

# Ensure you have set your OPENAI_API_KEY environment variable
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

def run_llm(prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
# # Example of Google Gemini API call
# import google.generativeai as genai
# import os

# # Ensure you have set your GOOGLE_API_KEY environment variable
# genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

# def run_llm(prompt):
#     model = genai.GenerativeModel("gemini-1.5-flash")
#     response = model.generate_content(
#         prompt,
#         generation_config=genai.types.GenerationConfig(
#             temperature=0
#         )
#     )
#     return response.text

In [ ]:
# Limit to top 5 for testing
eval_df = df.head(20).copy()

# Add LLM output to the dataframe
eval_df['LLM_output'] = eval_df['question'].apply(lambda q: run_llm(build_prompt(q, cot=True)))


In [34]:
eval_df

,question,answer,LLM_output
0,Jungkook is the 5th place. Find the number of ...,"If Jungkook is in 5th place, then 4 people cro...",To determine the number of people who crossed ...
1,A number divided by 10 is 6. Yoongi got the re...,"Let's call the certain number ""x"". According t...","To solve the problem, we need to find the numb..."
2,Dongju selects a piece of paper with a number ...,To find the second smallest and third smallest...,"To solve this problem, we need to determine al..."
3,"You wanted to subtract 46 from a number, but y...",If you accidentally subtracted 59 instead of 4...,"To solve this problem, let's break it down ste..."
4,The length of one span of Jinseo is about 12 c...,If one span of Jinseo is about 12 centimeters ...,"To solve the problem, we need to determine the..."
5,The owner of the snack shop said that for a sp...,"To receive the most sweets, Haneul should make...","To solve this problem, we need to form the lar..."
6,"For the natural number A, the quotient of A di...","To find the value of A, we can use the formula...","To solve the problem, we need to use the infor..."
7,How many diagonals can you draw in a decagon?,A decagon is a polygon with 10 sides. To find ...,To determine the number of diagonals in a deca...
8,What is the difference between the largest num...,"To find the largest number, we should arrange ...","To solve this problem, we need to form the lar..."
9,Find the sum of all multiples of 9 that are le...,To find the sum of all multiples of 9 that are...,To find the sum of all multiples of 9 that are...


In [ ]:
# Answer question by question in the df and print the LLM output as another column called 'llm_output' in the dataframe
results = []

for _, row in df.head(5).iterrows():  # limit for testing

    question = row['question']
    y = row['answer']

    prompt = build_prompt(question, cot=True)
    llm_output = run_llm(prompt)
    a = parse_answer(llm_output)

    if a is None:
        continue

    h0 = simulate_human(y)
    h1 = simulate_decision(h0, a)

    metrics = compute_metrics(h0, a, h1, y)

    results.append(metrics)

results_df = pd.DataFrame(results)
results_df.head()


In [30]:
metrics


{'human_correct': 1,
 'ai_correct': 0,
 'team_correct': 1,
 'ai_help': 0,
 'ai_harm': 0}

In [18]:
# Calculate overall performance
summary_metrics = results_df.mean()
print("--- Overall Evaluation Metrics ---")
print(summary_metrics)


"To solve the problem, we need to determine the length of the shorter side of the bookshelf in centimeters, given that it measures about two spans of Jinseo's hand.\n\n1. **Understand the measurement unit**: We are told that one span of Jinseo's hand is approximately 12 centimeters.\n\n2. **Determine the total length in spans**: The shorter side of the bookshelf is measured to be about two spans.\n\n3. **Calculate the length in centimeters**: Since one span is 12 centimeters, two spans would be:\n   \\[\n   2 \\text{ spans} \\times 12 \\text{ cm/span} = 24 \\text{ cm}\n   \\]\n\nTherefore, the length of the shorter side of the bookshelf is approximately 24 centimeters."